In [0]:
import requests

URL = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_week.geojson"
resp = requests.get(URL, timeout=30)
resp.raise_for_status()
features = resp.json()["features"]
print("Events fetched:", len(features))

In [0]:
from pyspark.sql import functions as F

def num(x):
    return float(x) if x is not None else None

rows = []
for f in features:
    p = f["properties"]
    lon, lat, depth = f["geometry"]["coordinates"]
    rows.append((
        f["id"], num(p.get("mag")), p.get("magType"), p.get("place"),
        p.get("time"), p.get("updated"),
        num(lat), num(lon), num(depth), p.get("type")
    ))

schema = """event_id string, magnitude double, mag_type string, place string,
            time_ms long, updated_ms long, latitude double, longitude double,
            depth_km double, event_type string"""

df = (spark.createDataFrame(rows, schema)
      .withColumn("event_time", (F.col("time_ms") / 1000).cast("timestamp"))
      .withColumn("ingested_at", F.current_timestamp()))

(df.write.mode("append")
   .saveAsTable("workspace.petropulse_bronze.usgs_seismic_raw"))

print("Rows written:", df.count())

In [0]:
%sql
SELECT COUNT(*) AS total_rows,
       MIN(event_time) AS oldest_event,
       MAX(event_time) AS newest_event,
       MAX(magnitude) AS max_magnitude
FROM workspace.petropulse_bronze.usgs_seismic_raw;